# Coding Homework 08: Matrix Norms and Conditioning

## Goal

Use matrix norms to answer two different questions:

1. **Size:** How large is a matrix, according to a chosen rule?
2. **Sensitivity:** How much can a matrix amplify an input or a relative error?

You will finish four notebook functions and add `spectral_normalize` to your
student-owned `mini_transformer.py`. No prior machine-learning knowledge is
assumed; every application term is defined when it appears.

**Estimated time:** 90–120 minutes
**Submit:** this notebook and `mini_transformer.py`


## Start here: how to work through this notebook

You do not need to remember an earlier programming course. Work one small step
at a time. The mathematics is the goal; the programming examples are here to
help you express it.

1. Save your own copy of the notebook. A **cell** is one block of text or code.
   Click a code cell and press **Shift+Enter** to run it. Run cells from the top
   down: later cells use names created by earlier cells.
2. Run the setup cell first. `import numpy as np` makes NumPy's array tools
   available under the short name `np`.
3. Read the explanation and run the small practice example before attempting
   the corresponding task. Practice cells are supplied; you may experiment
   with them, but restore them before the final run.
4. Edit only the requested task or response. `#` begins a comment. `pass` is a
   placeholder that does nothing; a function ending at `pass` returns `None`
   (no result). Replace it with your calculation and a `return` statement.
5. Running a cell containing `def` stores the function; it does not call it.
   After editing the definition, run that cell again, then run its check cell.
   A successful `assert` may print nothing. A skipped check is not a pass:
   if a result is still `None`, the task is unfinished.
6. If a check fails, stop there. Read the last line of the error, print the
   relevant array's `.shape`, and fix this task before running cells that need it.
   Errors below an unfinished task are expected and are not separate failures.
7. Save, restart the kernel/runtime (the Python session), and run everything
   from the beginning once your work is complete. This catches accidental use
   of a value left over from an earlier attempt.

**Reading code:** `=` stores a value; `==` compares values; `@` multiplies
matrices; `*` multiplies entries or scales an array; `**2` squares entries.
Python indices start at **0**: mathematical entry $a_{1,1}$ is `A[0, 0]`.
`A.shape` is `(number_of_rows, number_of_columns)`, `A.ndim` counts dimensions,
and `A.T` transposes a matrix. An ordinary vector has shape `(n,)`.

**If you get stuck:** record the function name, the smallest input that fails,
its shape, the expected mathematical result, and the last line of the error.
Include these when asking for help. You do not need to rewrite the whole cell.

**Plan your time:** if Python is new to you, allow extra time beyond the
headline estimate and work in several short sessions. Start with setup and
the practice example; then finish one task at a time.

More worked examples and an error-message checklist: [Python and NumPy help](`mt_utils.py` / `mini_transformer.py` in this folder).


### Which file do I edit?

Download the notebook and `mini_transformer.py` from this assignment's
`student` folder. Keep both in the same working folder. Use the supplied
completed starting model; you do not need to recover earlier submissions.

- **Notebook:** fill the named notebook functions and written responses.
- **`mini_transformer.py`:** open this plain-text Python file in an editor,
  find the named milestone function, replace its placeholder, and save.
  Keep the other supplied model functions. A `.py` file cannot see a function
  merely because you defined it in the notebook.
- **Colab:** use File → Upload notebook; in the Files panel upload the `.py`
  file. Double-click that file to edit it and save. If it is not editable in
  your interface, edit it locally and upload the replacement. Download both
  edited files before ending the session; Colab uploads are temporary.
- **Local Jupyter:** open the notebook from the folder containing both files.
  If setup reports `ModuleNotFoundError: mini_transformer`, check that folder
  and the spelling before trying to install anything.

Python remembers imported files. After saving a module edit, use the reload
cell beside the milestone (or restart and run from the top). Submit **both**
files to the course portal. Wait for the results and inspect the feedback; upload
alone does not confirm the work passed.


In [1]:
# Supplied practice: run this unchanged, then trace the result by hand.
import numpy as np

def double_and_total(values):
    values = np.asarray(values, dtype=float)  # turn a list into a numerical array
    doubled = 2 * values                     # one intermediate calculation
    total = np.sum(doubled)                   # add all entries
    return doubled, total                    # send two results back to the caller

practice_vector, practice_total = double_and_total([1, -2, 3])
print(practice_vector)        # [ 2. -4.  6.]
print(practice_vector.shape)  # (3,)
print(practice_total)         # 4.0
assert practice_total == 4.0


[ 2. -4.  6.]
(3,)
4.0


## Setup


In [2]:
import importlib
import numpy as np

import mini_transformer
importlib.reload(mini_transformer)

np.set_printoptions(precision=4, suppress=True)
print("Setup complete.")


Setup complete.


## 1. Entrywise size and induced norms

For a matrix $A=(a_{ij})$:

$$
\max_{i,j}|a_{ij}| \quad\text{is the largest absolute entry,}
$$

while

$$
\|A\|_1=\max_j\sum_i|a_{ij}|,\qquad
\|A\|_\infty=\max_i\sum_j|a_{ij}|.
$$

The induced $1$-norm is the largest absolute **column sum**. The induced
$\infty$-norm is the largest absolute **row sum**. Neither is generally the
largest entry. The Frobenius norm is
$\|A\|_F=(\sum_{i,j}a_{ij}^2)^{1/2}$.

Complete the function so it returns, in this order, the largest entry,
$\|A\|_1$, $\|A\|_\infty$, and $\|A\|_F$.


### Task plan: return four named calculations

The input checks are supplied. Compute four intermediate variables, then
return them in the requested order. Useful tools: `np.abs(A)` takes absolute
values entry by entry; `np.max(...)` chooses the largest; `np.sum(..., axis=0)`
adds down rows to give one total per **column**; `axis=1` adds across columns
to give one total per **row**. For the Frobenius norm, square, sum, and take
the square root with `np.sqrt` (or use `np.linalg.norm(A, ord="fro")`).

Try these on `[[2., -1.], [0., 3.]]`: absolute column sums are `[2., 4.]`;
absolute row sums are `[3., 3.]`. The largest entry is `3` but the 1-norm is `4`.
Print each intermediate quantity if the tuple check fails.


In [3]:
def entrywise_and_induced_norms(A):
    A = np.asarray(A, dtype=float)
    if A.ndim != 2 or 0 in A.shape:
        raise ValueError("A must be a nonempty matrix")
    # Compute the requested four norms
    largest_entry = np.max(np.abs(A))
    one_norm = np.max(np.sum(np.abs(A), axis=0))
    inf_norm = np.max(np.sum(np.abs(A), axis=1))
    fro_norm = np.linalg.norm(A, ord="fro")

    return largest_entry, one_norm, inf_norm, fro_norm

A_check = np.array([[1., -2., 0.], [3., 4., -1.]])
observed = entrywise_and_induced_norms(A_check)
if observed is not None:
    np.testing.assert_allclose(observed, (4., 6., 8., np.sqrt(31)))
    print("Entrywise/induced norms:", np.round(observed, 4))


Entrywise/induced norms: [4.     6.     8.     5.5678]


## 2. Norms built from singular values

If the singular values of $A$ are
$\sigma_1\geq\sigma_2\geq\cdots\geq0$, then

$$
\|A\|_2=\sigma_1,\qquad \|A\|_*=\sum_i\sigma_i.
$$

The spectral norm $\|A\|_2$ is the largest possible Euclidean amplification:

$$
\|Ax\|_2\leq\|A\|_2\|x\|_2.
$$

The nuclear norm $\|A\|_*$ adds all singular values and is different from the
spectral norm unless at most one singular value is nonzero.


### Task plan: one vector, two summaries

`np.linalg.svd(A, compute_uv=False)` returns only the singular-value vector,
in decreasing order. Store that vector; use index `0` for its largest value
and `.sum()` for its total. Return the two numbers in that order. This option
does **not** return `U, s, Vt`, so do not unpack it into three variables.


In [4]:
def singular_value_norms(A):
    A = np.asarray(A, dtype=float)
    if A.ndim != 2 or 0 in A.shape:
        raise ValueError("A must be a nonempty matrix")
    # Compute the singular values
    s = np.linalg.svd(A, compute_uv=False)

    # Spectral norm is the largest singular value
    spectral_norm = s[0]
    # Nuclear norm is the sum of all singular values
    nuclear_norm = s.sum()

    return spectral_norm, nuclear_norm

B_check = np.array([[3., 0.], [0., 2.], [0., 0.]])
observed = singular_value_norms(B_check)
if observed is not None:
    np.testing.assert_allclose(observed, (3., 5.))
    print("Spectral/nuclear norms:", observed)


Spectral/nuclear norms: (np.float64(2.9999999999999996), np.float64(5.0))


## 3. Condition number: a worst-case relative bound

For a full-rank matrix, the spectral condition number is

$$
\kappa_2(A)=\frac{\sigma_{\max}(A)}{\sigma_{\min}(A)}.
$$

A rank-deficient matrix has $\kappa_2(A)=\infty$. For an invertible square
system $Ax=b$, the condition number supplies the bound

$$
\frac{\|\delta x\|_2}{\|x\|_2}
\leq \kappa_2(A)\frac{\|\delta b\|_2}{\|b\|_2}.
$$

This is a **worst-case upper bound**. A large condition number does not say that
every small change in $b$ produces a large change in $x$; the perturbation's
direction matters.


### Task plan: handle numerical zero before dividing

Compute singular values, then compare the smallest with
`np.finfo(float).eps * max(A.shape) * singular_values[0]`. This supplied
threshold scales the computer's rounding precision to the matrix size.
If the smallest is at or below the threshold, return `np.inf`; otherwise
return the ratio in the formula. This convention treats numerically
indistinguishable-from-zero singular values as zero. It also handles the all-
zero matrix without dividing by zero. Input may be rectangular; “full rank”
here means rank equal to the smaller dimension.


In [5]:
def spectral_condition_number(A):
    A = np.asarray(A, dtype=float)
    if A.ndim != 2 or 0 in A.shape:
        raise ValueError("A must be a nonempty matrix")

    # Get singular values in decreasing order
    s = np.linalg.svd(A, compute_uv=False)

    # Calculate the threshold for numerical zero
    threshold = np.finfo(float).eps * max(A.shape) * s[0]

    # If the smallest singular value is at or below the threshold, it is rank-deficient
    if s[-1] <= threshold:
        return np.inf

    return s[0] / s[-1]

C_check = np.diag([4., 0.5])
if spectral_condition_number(C_check) is not None:
    assert np.isclose(spectral_condition_number(C_check), 8.0)
    assert np.isinf(spectral_condition_number(np.array([[1., 2.], [2., 4.]])))
    print("Condition-number checks passed.")


Condition-number checks passed.


## 4. Compare an actual solution change with the bound

Suppose $Ax=b$ and only the right-hand side changes, from $b$ to
$b+\delta b$. Complete the function below. It should solve both systems and
return

1. the actual relative solution change, and
2. the condition-number upper bound.

Require $A$ to be square and invertible, and require $b$ and $x$ to be nonzero
so the relative quantities are defined.


### Task plan: solve twice and compare

1. The supplied checks enforce `(n, n)`, `(n,)`, `(n,)` inputs. Reject zero `b`
   with `ValueError` because its relative change is undefined.
2. Call `spectral_condition_number(A)`. Require it to be finite; otherwise
   raise `ValueError` for this task's invertible-system requirement.
3. `np.linalg.solve(A, b)` returns the original solution. Call it again with
   `b + delta_b` for the changed solution. Do not form an inverse.
4. Subtract the two solutions; measure that vector and the original solution
   with `np.linalg.norm`. Reject a zero original solution before dividing.
5. Compute the actual relative change and the bound as **separate** numbers,
   following the displayed formula. Return them in that order.

The bound uses the relative change in `b`; the actual change uses the
relative change in `x`. Mixing these denominators is a common error.


In [6]:
def relative_solve_change(A, b, delta_b):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    delta_b = np.asarray(delta_b, dtype=float)
    if A.ndim != 2 or A.shape[0] != A.shape[1] or A.shape[0] == 0:
        raise ValueError("A must be a nonempty square matrix")
    if b.shape != (A.shape[0],) or delta_b.shape != b.shape:
        raise ValueError("b and delta_b must be matching vectors")

    # 1. Reject zero b because its relative change is undefined
    b_norm = np.linalg.norm(b)
    if np.isclose(b_norm, 0.0):
        raise ValueError("b must be a nonzero vector")

    # 2. Get condition number and ensure it is finite
    cond_A = spectral_condition_number(A)
    if np.isinf(cond_A):
        raise ValueError("A must be invertible (finite condition number)")

    # 3. Solve for original x and perturbed x_new
    x = np.linalg.solve(A, b)
    x_norm = np.linalg.norm(x)
    if np.isclose(x_norm, 0.0):
        raise ValueError("x must be a nonzero vector")

    x_new = np.linalg.solve(A, b + delta_b)
    delta_x = x_new - x

    # 4. Compute relative changes
    actual_change = np.linalg.norm(delta_x) / x_norm
    upper_bound = cond_A * (np.linalg.norm(delta_b) / b_norm)

    return actual_change, upper_bound

A_solve = np.diag([4., 0.5])
b_solve = np.array([4., 0.5])
delta_b = np.array([0., 0.05])
observed = relative_solve_change(A_solve, b_solve, delta_b)
if observed is not None:
    actual_change, upper_bound = observed
    print(f"Actual relative change: {actual_change:.5f}")
    print(f"Condition-number bound: {upper_bound:.5f}")
    assert actual_change <= upper_bound + 1e-12


Actual relative change: 0.07071
Condition-number bound: 0.09923


## 5. Mini-transformer milestone: spectral normalization

A transformer's **weight matrix** maps each token's feature row to a new
feature row. For example, its value projection is $V=XW_V$. If a token row is
perturbed by $e$, then its projected change satisfies

$$
\|eW_V\|_2\leq\|e\|_2\,\|W_V\|_2.
$$

**Spectral normalization** rescales a nonzero weight matrix to a chosen
spectral norm. It therefore controls this worst-case amplification. It does
not make the matrix well-conditioned: multiplying every singular value by the
same factor leaves their ratio unchanged.

Complete `spectral_normalize` in `mini_transformer.py`, then reload the module
and run the application check below.


### Model-file task: `spectral_normalize`

1. Open `mini_transformer.py` and locate `spectral_normalize`.
2. Convert `W` to a floating-point array and require a nonempty matrix.
   Require `target` to be a positive finite scalar; reject invalid inputs with
   `ValueError` or `TypeError`.
3. Compute the spectral norm with `np.linalg.norm(W, ord=2)`. Reject a zero
   norm: multiplying a zero matrix cannot make its norm positive.
4. Decide the scalar factor needed to change this norm to `target`. Multiply
   `W` by that factor and return a **new** array; do not change `W` in place.
5. Save the file. Run the next cell, which reloads it and checks the result.

For a matrix whose spectral norm is `6`, a target of `2` calls for a factor
of `1/3`. Scaling changes size; it does not change singular-value ratios.


### Check the model file before the full experiment

Open `mini_transformer.py` and search for **THIS WEEK**. The numbered plan and
tiny input/output example are now beside `spectral_normalize` itself. Complete that
function, save, then run the cell below. It reloads your file and checks a
small example before you encounter the longer model setup. An unfinished
function gives a message naming what to finish; do not skip this check.


In [7]:
import importlib
import mini_transformer as mt
mt = importlib.reload(mt)
mt.check_milestone()


HW11 project checks passed. Continue with the notebook experiment.


In [8]:
importlib.reload(mini_transformer)

W_value = np.array([
    [3.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 0.25],
])
W_controlled = mini_transformer.spectral_normalize(W_value, target=1.0)

assert np.isclose(np.linalg.norm(W_controlled, ord=2), 1.0)
assert np.isclose(np.linalg.cond(W_controlled), np.linalg.cond(W_value))

token_perturbation = np.array([0.02, -0.03, 0.01])
projected_change = token_perturbation @ W_controlled
bound = np.linalg.norm(token_perturbation) * np.linalg.norm(W_controlled, ord=2)
assert np.linalg.norm(projected_change) <= bound + 1e-12

print("Original spectral norm:", np.linalg.norm(W_value, ord=2))
print("Controlled spectral norm:", np.linalg.norm(W_controlled, ord=2))
print("Condition number (unchanged):", np.linalg.cond(W_controlled))
print("Observed projected change:", np.linalg.norm(projected_change))
print("Spectral-norm bound:", bound)


Original spectral norm: 3.0
Controlled spectral norm: 1.0
Condition number (unchanged): 12.0
Observed projected change: 0.022376202636829254
Spectral-norm bound: 0.03741657386773942


## 6. Interpretation and submission check

In the Markdown cell below, answer briefly:

1. Why can the matrix infinity-norm be larger than every individual entry?
2. What does a large condition number say—and what does it **not** say?
3. Why does spectral normalization control maximum amplification without
   changing the condition number?

Before submitting:

- [ ] Restart the kernel and run all cells.
- [ ] Confirm every assertion passes.
- [ ] Submit the notebook and `mini_transformer.py` together.
- [ ] If you used an AI assistant, state what you asked it and what you checked
  or changed.


Response: The matrix infinity-norm can be larger than every entry because it measures the largest absolute row sum, not the largest single entry. A large condition number means the matrix is sensitive to perturbations in some directions, but it does not mean every input is equally sensitive. Spectral normalization rescales all singular values by the same factor, so it limits the maximum amplification while leaving their ratio unchanged.


### Your response

1.
2.
3.

**AI-use note (write “none” if not used):**
